In [1]:
import pennylane as qml
from pennylane import numpy as np

## Circuits as functions

In many modern quantum applications (Quantum Machine Learning and Quantum Chemistry), it is possible to see quantum circuits as mathematical models or functions.

For instance, a circuit with four qubits and composed of parametric gates (i.e. gates that depend on a value theta) and whose output is the expectation value of the first qubit under $Z$, can be seen as the function:

\begin{align*}
    F : \mathbb{R}^4 & \rightarrow \mathbb{R} \\
    (\theta_0, \theta_1, \theta_2, \theta_3) & \mapsto \braket{Z_0}
\end{align*}

where the coordinates $(\theta_1, \dots, \theta_3)$ are the gate parameters and $\braket{Z_0}$ is the expectation value of the Pauli-Z observable on the first wire (i.e. on the first qubit)


PennyLane has some built-in templates that allows us to build expressive quantum circuits:
1. **`qml.BasicEntanglerLayers`**: 
    * layers consisting of one-parameter single-qubit rotations on each qubit, followed by a closed chain of $CNOT$ gates
    * the $CNOT$ gates connects every qubit to its neighbour 
    * by pre-definition the rotation gate is **RX**

In [2]:
n_wires = 4
dev = qml.device("default.qubit", wires=n_wires)

@qml.qnode(dev)
def entangler_circuit(weights, rot):
    qml.BasicEntanglerLayers(weights, wires=range(n_wires), rotation=rot)
    return qml.expval(qml.Z(0))

#entangler-circuit with only one layer
print("Entangler-circuit with a single layer")
print(qml.draw(entangler_circuit, level="device")([[0.1, 0.2, 0.3, 0.4]], None))

print("\n")

#entangler-circuit with two layers
print("Entangler-circuit with two layers")
print(qml.draw(entangler_circuit, level="device")([[0.1, 0.2, 0.3, 0.4],[0.5, 0.6, 0.7, 0.8]], None))

print("\n")

#entangler-circuit wiht one layer and rotation gate RZ
print("Entangler-circuit with one layer and rotation gate RZ")
print(qml.draw(entangler_circuit, level="device")([[0.1, 0.2, 0.3, 0.4]], qml.RZ))

Entangler-circuit with a single layer
0: ──RX(0.10)─╭●───────╭X─┤  <Z>
1: ──RX(0.20)─╰X─╭●────│──┤     
2: ──RX(0.30)────╰X─╭●─│──┤     
3: ──RX(0.40)───────╰X─╰●─┤     


Entangler-circuit with two layers
0: ──RX(0.10)─╭●───────╭X──RX(0.50)─╭●───────╭X─┤  <Z>
1: ──RX(0.20)─╰X─╭●────│───RX(0.60)─╰X─╭●────│──┤     
2: ──RX(0.30)────╰X─╭●─│───RX(0.70)────╰X─╭●─│──┤     
3: ──RX(0.40)───────╰X─╰●──RX(0.80)───────╰X─╰●─┤     


Entangler-circuit with one layer and rotation gate RZ
0: ──RZ(0.10)─╭●───────╭X─┤  <Z>
1: ──RZ(0.20)─╰X─╭●────│──┤     
2: ──RZ(0.30)────╰X─╭●─│──┤     
3: ──RZ(0.40)───────╰X─╰●─┤     


2. **`qml.StronglyEntanglingLayers`**: 
    * increases the level of entanglement between all qubits
    * it has more parameters than *qml.BasicEntanglerLayers* 
    * by pre-definition the rotation gate is **RX**

In [3]:
@qml.qnode(dev)
def strong_entangler_circuit(weights, ran, impr):
    qml.StronglyEntanglingLayers(weights, wires=range(n_wires), ranges=ran, imprimitive=impr)
    return qml.expval(qml.Z(0)) 
    #return [qml.expval(qml.Z(i)) for i in range(n_wires)]

#randomly create parameters for the rotation gates, with a certain number of layers
shape = qml.StronglyEntanglingLayers.shape(n_layers=1, n_wires=n_wires)
rng = np.random.default_rng(12345)
weights = rng.random(size=shape)

print("Strongly-entangler-circuit with single layer")
print(qml.draw(strong_entangler_circuit, level="device")(weights, None , None))

print("\n")
print("Strongly-entangler-circuit with single layer and changing the range option")
print(qml.draw(strong_entangler_circuit, level="device")(weights, [2] , None))

print("\n")
print("Strongly-entangler-circuit with single layer, changing the range option, and the controlled gate")
print(qml.draw(strong_entangler_circuit, level="device")(weights, [3] , qml.CZ))

Strongly-entangler-circuit with single layer
0: ──Rot(0.23,0.32,0.80)─╭●───────╭X─┤  <Z>
1: ──Rot(0.68,0.39,0.33)─╰X─╭●────│──┤     
2: ──Rot(0.60,0.19,0.67)────╰X─╭●─│──┤     
3: ──Rot(0.94,0.25,0.95)───────╰X─╰●─┤     


Strongly-entangler-circuit with single layer and changing the range option
0: ──Rot(0.23,0.32,0.80)─╭●────╭X────┤  <Z>
1: ──Rot(0.68,0.39,0.33)─│──╭●─│──╭X─┤     
2: ──Rot(0.60,0.19,0.67)─╰X─│──╰●─│──┤     
3: ──Rot(0.94,0.25,0.95)────╰X────╰●─┤     


Strongly-entangler-circuit with single layer, changing the range option, and the controlled gate
0: ──Rot(0.23,0.32,0.80)─╭●─╭Z───────┤  <Z>
1: ──Rot(0.68,0.39,0.33)─│──╰●─╭Z────┤     
2: ──Rot(0.60,0.19,0.67)─│─────╰●─╭Z─┤     
3: ──Rot(0.94,0.25,0.95)─╰Z───────╰●─┤     


## Gradient of a Circuit

Let us consider again the circuit that is represented by the function:
\begin{align*}
    F : \mathbb{R}^4 & \rightarrow \mathbb{R} \\
    (\theta_0, \theta_1, \theta_2, \theta_3) & \mapsto \braket{Z_0}
\end{align*}

Derivatives of functions are useful in optimization tasks.

Since quantum circuits can be seen as functions, we can try to calculate its derivates.
This is useful in many optimzation-related applications in Quantum Chemistry and Quantum Machine Learning

The derivate of $F$ is given by its gradient, which is the vector
\begin{align*}
    \triangledown F = \left( 
            \dfrac{\partial F}{\partial \theta_0},
            \dfrac{\partial F}{\partial \theta_1},
            \dfrac{\partial F}{\partial \theta_2},
            \dfrac{\partial F}{\partial \theta_3}
         \right)
\end{align*}

To find this gradient, we resort to **parameter shift-rules**.
The simplest parameter shift rule states that, when $F$ represents the expectation value of a quantum circuit with only single-parameter gates then
\begin{align*}
    \dfrac{\partial F}{\partial \theta_i} = \dfrac{1}{2} \left[
            F \left(\theta_i + \dfrac{\pi}{2} \right) - F \left(\theta_i - \dfrac{\pi}{2} \right)
        \right]
\end{align*}

There exists generalizations for cases in which gates are multi-parametric.
Such parameters shift-rules are advantageous since they allow to compute gradients with little overhead and in an error-robust manner.

To find the gradient of a circuit we resort to **`qml.jacobian`**

In order to execute such task we need to take into consideration some details:
1. We must tell PennyLane that the arguments are differentiable. Thus, when setting the input parameters, we must also include `requires_grad=True`
2. When defining the qnode, we must fill the `diff_method` (the pre-defined setting is to use standard numerical methods)

So, instead of doing the following:

In [4]:
@qml.qnode(dev) #qnode uses standard numerical methods
def circ1(weights):
    qml.BasicEntanglerLayers(weights, wires=range(n_wires))
    return qml.expval(qml.Z(0))

test_weights = np.array([[0.1, 0.2, 0.3, 0.4]]) #it is not indicated that the arguments are differentiable

print(qml.jacobian(circ1)(test_weights))

[[ 1.38777878e-17 -1.74813749e-01 -2.66766415e-01 -3.64609810e-01]]


We should do the following:

In [5]:
@qml.qnode(dev, interface="autograd", diff_method="parameter-shift") #qnode uses the parameter-shift differential method
#@qml.qnode(dev) #qnode uses standard numerical methods
def circ2(weights):
    qml.BasicEntanglerLayers(weights, wires=range(n_wires))
    return qml.expval(qml.Z(0))

test_weights = np.array([[0.1, 0.2, 0.3, 0.4]], requires_grad=True) #it is indicated that the arguments are differentiable
#test_weights = np.array([[0.1, 0.2, 0.3, 0.4]]) #it is not indicated that the arguments are differentiable

print(qml.jacobian(circ2)(test_weights))

[[ 0.         -0.17481375 -0.26676641 -0.36460981]]


From the experiences done, it suffices to indicate the `diff_method` (do not know what the parameter `interface` does)

However, to be sure that everyhting works as it should, we must follow the recipe provided by the PennyLane tutorial

There could be situations in which we may only want some parameters of a circuit to be differentiable, while others should remain fixed.
This is the reason why `requires_grad` is important.
For example, let us consider two basic entangler layers and differentiate only w.r.t the parameters `diff_weights` in the first layer.
The parameters `fixed_weight` in the second layer remain constant.

In [6]:
@qml.qnode(dev, interface="autograd", diff_method="parameter-shift")
def circ3(diff_weights, fixed_weights):
    qml.BasicEntanglerLayers(diff_weights, wires=range(n_wires))
    qml.BasicEntanglerLayers(fixed_weights, wires=range(n_wires))
    return qml.expval(qml.Z(0))

test_diff_weights = np.array([[0.5,0.1,-0.4,0.6]], requires_grad = True)
test_fixed_weights = np.array([[0.1,0.2,0.3,0.4]], requires_grad = False)

print(qml.jacobian(circ3)(test_diff_weights, test_fixed_weights))

[[-0.30647646  0.06160872  0.         -0.41303853]]


## Jacobian of a Circuit

Until this point, we only considered the case where the output of a quantum circuit is given by the expectation value of an observable.
However, that may not always be the case.

Whenever the output of a quantum circuit, with $k$ wires, are measurement probabilities, the output can be a real-value vector $(F_0, \dots, F_{m-1})$, with $m=2^k$ components.
If the circuit depends on $n$ gate parameters $(\theta_0, \theta_1, \dots, \theta_{n-1})$, the circuit can be interpreted as a function:
\begin{align*}
    F : \mathbb{R}^n & \rightarrow \mathbb{R}^m \\
    (\theta_0, \dots, \theta_{n-1}) & \mapsto (F_0, \dots, F_{m-1})
\end{align*}

In this case, the derivate of $F$ is represented by the $m \times n$ Jacobian matrix (insert here later).

`qml.jacobian` calculates the gradient of a circuit, but also its Jacobian matrix.
To obtain the latter, we must pass the gate parameters as a vector together with `requires_grad=True`.

In [7]:
@qml.qnode(dev)
def circ4(params):
    qml.RX(params[0], 0)
    qml.CNOT([0,1])
    qml.RY(params[1],0)
    return qml.probs([0,1])

sample_params = np.array([0.1, 0.2], requires_grad=True)
print(qml.jacobian(circ4)(sample_params))

[[-0.0494192  -0.09908654]
 [ 0.00049751  0.00024813]
 [-0.00049751  0.09908654]
 [ 0.0494192  -0.00024813]]


## Higher-order derivatives

In optimization applications, we may be interested in finding second derivatives.
First-order derivatives provides us information about the rate change of a function:
* slope of the tangent line at a point
    * f'(x) > 0: the slope is positive and the function is increasing
    * f'(x) < 0: the slope is negative and the function is decreasing
    * f'(x) = 0: the slope is null; critical point (maximum, minimum, or saddle point)
        * maximum: function decreases in all directions
        * minimum: function increases in all directions
        * saddle: function increases in a direction, and decreases in another direction
* rate of change in any direction
* first-order (linear) approximation of the function

Second-order derivatives provides us information on hoe the rate change itself is changing:
* curvature/concavity of the graph
    * f''(x) > 0: graph is concave up (smilling)
    * f''(x) < 0: graph is concave down (frowning)
    * f''(x) = 0: possible inflection point (change in concavity)
* At a point where f'(x) = 0:
    * f''(x) > 0: local minimum
    * f''(x) < 0: local maximum
    * f''(x) = 0: inconclusive

Let us return to $F: \mathbb{R}^n \rightarrow \mathbb{R}$, where we are interested on the stationary points, i.e. critical points, in whuch the gradient is zero.
These points may be local or global minima, maxima, saddles, or inflections of $F$.
To known which type of point this point is we resort to the Hessian matrix, which contains the second partial derivatives of $F$ (insert matrix later):
\begin{align*}
 H[F] = \dots
\end{align*}

When this matrix is:
* positive definite (symmetric and all eigenvalues are positive, i.e. greater than zero), we have a global minimum
* negative definite (symmetric and all eigenvalues are negative), we have a ?maximum?
* looking at specific eigenvalues can inform us about saddle or inflections points

The Hessian can be calculated as the Jacobian of the gradient of $F$:
$$
    H[F] = J[\triangledown F]
$$

We can resort do PennyLane do to such task.
For that, we need to set `max_diff` to 2, to inform PennyLane that we want to differentiate the QNode twice.
The default value of `max_diff` is 1, and failing to set it to the number of derivatives we want to apply may produce errors.

In [8]:
@qml.qnode(dev, diff_method="parameter-shift", max_diff=2)
def circ5(params):
    qml.RX(params[0], 0)
    qml.CNOT([0,1])
    qml.RY(params[1],0)
    return qml.expval(qml.Z(0))

test_params = np.array([0.7,0.3], requires_grad = True)
qml.jacobian(qml.jacobian(circ5))(test_params)

array([[-0.73068165,  0.19037934],
       [ 0.19037934, -0.73068165]])

## Optmizing circuits

From `circ5`, we would like to know what is the minimum expectation value that the circuit output $\braket{Z_0}$ can have.
For that we treat `circ5` as a **cost function**, i.e. a function we would like to optimize.
For such task, we need to use *optimizers*. The most famous and simplest optimizer is **Gradient Descent** (note that PennyLane has many built-in optimizers).

For this example we shall use `qml.GradientDescentOptimizer`, which has a `stepsize` argument usually known as *learning rate*.

There is a number of steps involved in optimizing, so we will define the following `optimize` function that can be easily modified into more advanced implementations. The `optimize` routine will take in a `cost_function`, the initial parameters `init_parameters` for the optimizer, and a number of `steps` (if needed).

In [16]:
def optimize(cost_function, init_params, *steps):
    opt = qml.GradientDescentOptimizer(stepsize=0.4)
    steps = 100
    params = init_params

    for i in range(steps):
        params = opt.step(cost_function, params)

    #returns the parameters for which cost_function is optimized and the minimum value of cost_function
    return params, cost_function(params)


initial_parameters = np.array([0.5,0.5], requires_grad = True)
print(optimize(circ5, initial_parameters, 100))

(tensor([1.57079633, 1.57079633], requires_grad=True), array(0.))
